# [재학습 · 2026-10-08] β=30 시드 16개 비중평균 앙상블 (6.4절)

이미 학습한 β=30 모델 16개(배포 시드 42 + 기존 5개 + 신규 10개)를 불러와 비중평균 앙상블을 검증·테스트 구간에서 평가한다. **학습 없음.** 원본 `crypto_beta_robustness_colab`(커밋 `0e86a41`)의 준비 셀과 앙상블 함수를 그대로 쓰고 모델 집합만 넓혔다.

- **`런타임 > 모두 실행`만 누르면 된다.** CPU로 충분, 약 20~40분.
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과: `data/crypto/rt1006/crypto_beta_ensemble16.csv` (16개 전부 / 배포 시드 제외 15개 × valid·test).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = 'rt1006'
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


In [ ]:
# 준비: 데이터·환경·평가 함수 (β=30으로 치환) + 기존 시드 5개 로드
run_cells('crypto/notebooks/crypto_beta_robustness_colab.ipynb', [3, 5, 6, 8, 10, 12, 13, 15, 17, 20], patch_after=3, extra="BETA = 30   # 재학습 후 검증 샤프 최고 β\nPOOL_DIR_ROBUST = MODEL_DIR / 'crypto_beta_robustness'\nDEPLOYED_AGENT_PATH = MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{BETA}.zip'\nassert DEPLOYED_AGENT_PATH.exists(), f'재학습한 v4 β={BETA} 모델 없음: {DEPLOYED_AGENT_PATH}'\nprint(f'[β 변경] BETA={BETA} | 기준 모델: {DEPLOYED_AGENT_PATH}')\n")


In [ ]:
# 배포 시드 42 + 기존 5개 + 신규 10개 모델 로드 (학습 없음)
deployed_model = PPO.load(str(DEPLOYED_AGENT_PATH))
SEEDS_EXTRA = [66, 77, 88, 99, 110, 121, 132, 143, 154, 165]
seed_models_extra = {s: PPO.load(str(POOL_DIR_ROBUST / f'agent_beta{BETA}_seed{s}.zip')) for s in SEEDS_EXTRA}
print('로드 완료: 42 +', list(seed_models.keys()), '+', list(seed_models_extra.keys()))


In [ ]:
# 원본 §6.4의 run_ensemble_smoothed 정의 셀
run_cells('crypto/notebooks/crypto_beta_robustness_colab.ipynb', [29], patch_after=None)


In [ ]:
%%time
# 16시드 비중평균 앙상블 — run_ensemble_smoothed(원본 §6.4 셀)를 그대로 쓰고 모델 집합만 넓힌다
ALL16 = {42: deployed_model, **seed_models, **seed_models_extra}
NO42 = {k: v for k, v in ALL16.items() if k != 42}
rows = {}
for name, models in [('앙상블(16개 전부)', ALL16), ('앙상블(배포 시드 제외 15개)', NO42)]:
    for split, bk in [('valid', valid_buckets), ('test', test_buckets)]:
        print(f'{name} {split} 실행 중...')
        vals, turns = run_ensemble_smoothed(models, ALPHA, bk)
        rows[(name, split)] = perf_row(vals, turns)
ens16 = pd.DataFrame(rows).T
print(); print(ens16.to_string(float_format=lambda x: f'{x:.4f}'))
for name in ['앙상블(16개 전부)', '앙상블(배포 시드 제외 15개)']:
    v, t = ens16.loc[(name, 'valid'), 'Sharpe'], ens16.loc[(name, 'test'), 'Sharpe']
    print(f'{name}: valid Sharpe {v:.3f} | test Sharpe {t:.3f} | 격차 {abs(t - v):.3f}')
ens16.to_csv(CRYPTO_DIR / 'crypto_beta_ensemble16.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_beta_ensemble16.csv')
